# Congress.gov API Explorer

This notebook provides flexible utilities to call Congress.gov API for any URL and explore the results.

## Features
- Call any Congress.gov API endpoint
- Parse and display results
- Save results to JSON
- Pretty-print API responses
- Handle pagination and rate limiting

In [ ]:
import json
import os
from pathlib import Path
from typing import Any, Dict, List, Optional
from urllib.parse import urlencode, parse_qs, urlparse
from urllib.request import Request, urlopen
import time
from datetime import datetime
from dataclasses import dataclass, asdict

# Get API key from environment
# Preferred: export CONGRESS_API_KEY in ~/.zshrc or ~/.bashrc
# Fallback: will attempt to load from .env if present

CONGRESS_API_KEY = os.getenv('CONGRESS_API_KEY', '')

if not CONGRESS_API_KEY:
    # Try to load from .env as fallback (if it exists)
    try:
        from dotenv import load_dotenv
        env_paths = [Path('.env'), Path('../.env'), Path('../../.env')]
        for env_path in env_paths:
            if env_path.exists():
                load_dotenv(env_path)
                CONGRESS_API_KEY = os.getenv('CONGRESS_API_KEY', '')
                if CONGRESS_API_KEY:
                    print(f"✓ CONGRESS_API_KEY loaded from {env_path.resolve()}")
                    break
    except ImportError:
        pass
else:
    print("✓ CONGRESS_API_KEY loaded from environment")

if not CONGRESS_API_KEY:
    print("⚠️  WARNING: CONGRESS_API_KEY not found")
    print("  Set it in your shell profile:")
    print("  export CONGRESS_API_KEY='your_key_here'")

print(f"Working directory: {Path.cwd()}")

## Core API Client

In [ ]:
@dataclass
class APIResponse:
    """Wrapper for Congress.gov API responses."""
    url: str
    params: Dict[str, Any]
    status_code: int
    data: Dict[str, Any]
    error: Optional[str] = None
    timestamp: str = None
    
    def __post_init__(self):
        if not self.timestamp:
            self.timestamp = datetime.utcnow().isoformat()


class CongressAPIClient:
    """Flexible client for Congress.gov API."""
    
    BASE_URL = "https://api.congress.gov/v3"
    DEFAULT_TIMEOUT = 30
    
    def __init__(self, api_key: str = None, rate_delay: float = 0.5):
        """Initialize client.
        
        Args:
            api_key: Congress.gov API key (uses CONGRESS_API_KEY env var if not provided)
            rate_delay: Delay between requests in seconds (default 0.5s)
        """
        self.api_key = api_key or CONGRESS_API_KEY
        self.rate_delay = rate_delay
        self.last_request_time = 0
    
    def _enforce_rate_limit(self):
        """Enforce rate limiting between requests."""
        elapsed = time.time() - self.last_request_time
        if elapsed < self.rate_delay:
            time.sleep(self.rate_delay - elapsed)
    
    def call(self, path: str, params: Dict[str, Any] = None) -> APIResponse:
        """Call Congress.gov API endpoint.
        
        Args:
            path: API path (e.g., '/bill' or '/bill/119/hr/1')
            params: Query parameters (optional)
        
        Returns:
            APIResponse object with data and metadata
        """
        if not self.api_key:
            return APIResponse(
                url='',
                params=params or {},
                status_code=401,
                data={},
                error='CONGRESS_API_KEY not set'
            )
        
        self._enforce_rate_limit()
        
        # Build URL
        url = f"{self.BASE_URL}{path}"
        query_params = params.copy() if params else {}
        query_params['api_key'] = self.api_key
        query_params['format'] = 'json'
        
        query_string = urlencode(query_params)
        full_url = f"{url}?{query_string}"
        
        try:
            self.last_request_time = time.time()
            
            req = Request(
                full_url,
                headers={
                    'Accept': 'application/json',
                    'User-Agent': 'congress-api-explorer/1.0'
                }
            )
            
            with urlopen(req, timeout=self.DEFAULT_TIMEOUT) as resp:
                data = json.loads(resp.read().decode('utf-8'))
                return APIResponse(
                    url=url,
                    params=query_params,
                    status_code=200,
                    data=data
                )
        except json.JSONDecodeError as e:
            return APIResponse(
                url=url,
                params=query_params,
                status_code=500,
                data={},
                error=f'JSON decode error: {str(e)}'
            )
        except Exception as e:
            return APIResponse(
                url=url,
                params=query_params,
                status_code=500,
                data={},
                error=str(e)
            )
    
    def call_url(self, full_url: str) -> APIResponse:
        """Call a full Congress.gov API URL directly.
        
        Args:
            full_url: Full URL including query parameters
        
        Returns:
            APIResponse object
        """
        # Parse URL to extract path and params
        parsed = urlparse(full_url)
        path = parsed.path.replace('/v3', '')  # Remove /v3 if present
        
        # Parse existing query params
        query_params = {}
        if parsed.query:
            for key, values in parse_qs(parsed.query).items():
                query_params[key] = values[0] if values else ''
        
        # Remove api_key if present (we'll add it back)
        query_params.pop('api_key', None)
        query_params.pop('format', None)
        
        return self.call(path, query_params)


# Initialize client
client = CongressAPIClient()
print("✓ CongressAPIClient initialized")

## Utility Functions

In [ ]:
def save_response(response: APIResponse, output_file: str = None) -> str:
    """Save API response to JSON file.
    
    Args:
        response: APIResponse object
        output_file: Output file path (optional, auto-generated if not provided)
    
    Returns:
        Path to saved file
    """
    if not output_file:
        timestamp = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
        output_file = f"congress_api_response_{timestamp}.json"
    
    output_path = Path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(output_path, 'w') as f:
        json.dump(asdict(response), f, indent=2)
    
    print(f"✓ Saved to {output_path.resolve()}")
    return str(output_path.resolve())


def pretty_print(data: Any, max_depth: int = 3, current_depth: int = 0):
    """Pretty-print JSON data with depth limiting.
    
    Args:
        data: Data to print
        max_depth: Maximum nesting depth to display
        current_depth: Current recursion depth
    """
    indent = '  ' * current_depth
    
    if current_depth >= max_depth:
        if isinstance(data, dict):
            print(f"{indent}{{...{len(data)} keys}}")
        elif isinstance(data, list):
            print(f"{indent}[...{len(data)} items]")
        else:
            print(f"{indent}{data}")
        return
    
    if isinstance(data, dict):
        print(f"{indent}{{")
        for key, value in list(data.items())[:10]:  # Limit keys shown
            print(f"{indent}  {key}: ", end="")
            if isinstance(value, (dict, list)):
                print()
                pretty_print(value, max_depth, current_depth + 2)
            else:
                print(repr(value))
        if len(data) > 10:
            print(f"{indent}  ... and {len(data) - 10} more keys")
        print(f"{indent}}}")
    elif isinstance(data, list):
        print(f"{indent}[")
        for item in data[:5]:  # Show first 5 items
            pretty_print(item, max_depth, current_depth + 1)
        if len(data) > 5:
            print(f"{indent}  ... and {len(data) - 5} more items")
        print(f"{indent}]")
    else:
        print(f"{indent}{repr(data)}")


def count_items(data: Any, key: str = None) -> int:
    """Count items in API response.
    
    Args:
        data: Response data
        key: Specific key to count (optional)
    
    Returns:
        Item count
    """
    if key:
        items = data.get(key, [])
    else:
        # Try common keys
        for k in ['bills', 'members', 'committees', 'amendments', 'items']:
            if k in data:
                items = data[k]
                break
        else:
            items = []
    
    return len(items) if isinstance(items, list) else 0


print("✓ Utility functions loaded")

## Example 1: Fetch Bills (Basic)

In [ ]:
# Example: Fetch recent bills
response = client.call(
    '/bill',
    params={
        'fromDateTime': '2026-01-01T00:00:00Z',
        'limit': 10
    }
)

print(f"Status: {response.status_code}")
print(f"Timestamp: {response.timestamp}")
print(f"Items returned: {count_items(response.data, 'bills')}")
print()
print("Response structure:")
pretty_print(response.data)

## Example 2: Fetch Specific Bill

In [ ]:
# Example: Fetch a specific bill
# Format: /bill/{congress}/{billType}/{billNumber}
response = client.call('/bill/119/hr/1')

print(f"Status: {response.status_code}")
if response.error:
    print(f"Error: {response.error}")
else:
    bill = response.data.get('bill', {})
    if bill:
        print(f"Title: {bill.get('title')}")
        print(f"Type: {bill.get('type')}")
        print(f"Number: {bill.get('number')}")
        print(f"Congress: {bill.get('congress')}")
    print()
    print("Full response:")
    pretty_print(response.data)

## Example 3: Fetch Members of Congress

In [ ]:
# Example: Fetch current members of Congress
response = client.call(
    '/member',
    params={'limit': 5}
)

print(f"Status: {response.status_code}")
print(f"Members returned: {count_items(response.data, 'members')}")
print()
if response.data.get('members'):
    print("Sample members:")
    for member in response.data['members'][:3]:
        print(f"  - {member.get('name')} ({member.get('party')})")
print()
print("Response structure:")
pretty_print(response.data)

## Example 4: Fetch Committees

In [ ]:
# Example: Fetch committees
response = client.call(
    '/committee',
    params={'limit': 5}
)

print(f"Status: {response.status_code}")
print(f"Committees returned: {count_items(response.data, 'committees')}")
print()
if response.data.get('committees'):
    print("Sample committees:")
    for committee in response.data['committees'][:3]:
        print(f"  - {committee.get('name')}")
print()
print("Response structure:")
pretty_print(response.data)

## Custom Query: Use Your Own URL

In [ ]:
# Replace with your desired Congress.gov API URL
# Examples:
# - https://api.congress.gov/v3/bill?api_key=...&limit=10
# - https://api.congress.gov/v3/amendment/119/s/1
# - https://api.congress.gov/v3/summaries?api_key=...&limit=5

CUSTOM_URL = "https://api.congress.gov/v3/bill?fromDateTime=2026-02-01T00:00:00Z&limit=5"

response = client.call_url(CUSTOM_URL)

print(f"Status: {response.status_code}")
if response.error:
    print(f"Error: {response.error}")
else:
    print(f"URL: {response.url}")
    print(f"Items returned: {count_items(response.data)}")
    print()
    print("Response preview:")
    pretty_print(response.data, max_depth=2)

## Save and Export

In [ ]:
# Save the most recent response
# output_file parameter is optional; if not provided, a timestamped filename is created

# Save to specific location
saved_path = save_response(response, 'congress_api_response.json')
print(f"Response saved to: {saved_path}")

# Or let it auto-generate the filename
# saved_path = save_response(response)
# print(f"Response saved to: {saved_path}")

## API Endpoint Reference

Common Congress.gov API v3 endpoints:

### Bills
- `GET /bill` - List bills
- `GET /bill/{congress}/{billType}/{billNumber}` - Get specific bill
- `GET /bill/{congress}/{billType}/{billNumber}/amendments` - Amendments to a bill
- `GET /bill/{congress}/{billType}/{billNumber}/committees` - Bill committees
- `GET /bill/{congress}/{billType}/{billNumber}/relatedbills` - Related bills
- `GET /bill/{congress}/{billType}/{billNumber}/text` - Bill text versions
- `GET /bill/{congress}/{billType}/{billNumber}/actions` - Bill actions/history

### Members
- `GET /member` - List members
- `GET /member/{bioguideId}` - Get specific member

### Committees
- `GET /committee` - List committees
- `GET /committee/{congress}/{chamber}/{committeeCode}` - Get specific committee

### Amendments
- `GET /amendment` - List amendments
- `GET /amendment/{congress}/{amendmentType}/{amendmentNumber}` - Get specific amendment

### Other
- `GET /summaries` - Bill summaries
- `GET /congress` - Congress sessions

### Query Parameters
- `fromDateTime`: Filter from date (ISO 8601 format)
- `toDateTime`: Filter to date
- `limit`: Number of results (default 20, max depends on endpoint)
- `offset`: Pagination offset
- `sort`: Sort field (varies by endpoint)

**Documentation:** https://github.com/LibraryOfCongress/api.congress.gov

## Advanced: Pagination Helper

In [ ]:
def fetch_all_pages(path: str, params: Dict[str, Any] = None, max_pages: int = None, items_key: str = 'bills') -> List[Any]:
    """Fetch all pages of results from an endpoint.
    
    Args:
        path: API path
        params: Query parameters
        max_pages: Maximum pages to fetch (None = no limit)
        items_key: Key in response containing items list
    
    Returns:
        List of all items across all pages
    """
    all_items = []
    offset = 0
    page = 0
    
    if params is None:
        params = {}
    else:
        params = params.copy()
    
    while True:
        if max_pages and page >= max_pages:
            print(f"Reached max_pages limit: {max_pages}")
            break
        
        params['offset'] = offset
        response = client.call(path, params)
        
        if response.status_code != 200:
            print(f"Error on page {page}: {response.error}")
            break
        
        items = response.data.get(items_key, [])
        if not items:
            print(f"No more items at page {page}")
            break
        
        all_items.extend(items)
        print(f"Page {page}: {len(items)} items (total: {len(all_items)})")
        
        if len(items) < (params.get('limit', 20)):
            # Last page has fewer items than limit
            print("Reached last page")
            break
        
        offset += len(items)
        page += 1
    
    return all_items


print("✓ Pagination helper loaded")

## Example: Fetch All Bills from a Date Range

In [ ]:
# Fetch multiple pages of bills
bills = fetch_all_pages(
    '/bill',
    params={
        'fromDateTime': '2026-02-01T00:00:00Z',
        'limit': 20
    },
    max_pages=3,  # Limit to 3 pages for demo
    items_key='bills'
)

print(f"\nTotal bills fetched: {len(bills)}")
if bills:
    print("\nSample bills:")
    for bill in bills[:5]:
        print(f"  - {bill.get('number')} ({bill.get('type').upper()}): {bill.get('title')}")